# EDA: Basic Dataset Overview

**Goal**: Load datasets and perform basic sanity checks.

## 1. Setup — Import libraries

In [1]:
import os
import json
import glob
import pandas as pd
# matplotlib 

## 2. List all JSON files in data/raw/

In [2]:
# Setup: Download dataset
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q kaggle
    
    # Kaggle credentials
    import json
    from getpass import getpass
    kaggle_username = input("Kaggle username: ")
    kaggle_key = getpass("Kaggle API key: ")
    
    !mkdir -p ~/.kaggle
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": kaggle_username, "key": kaggle_key}, f)
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Download and extract
    !kaggle datasets download -d incidelen/medturkquad -p /content -q
    !unzip -o -q /content/medturkquad.zip -d /content
    !mkdir -p /content/data/raw/MedTurkQuAD
    
    import shutil
    for json_file in glob.glob('/content/*.json'):
        shutil.move(json_file, '/content/data/raw/MedTurkQuAD/')
    !rm -f /content/medturkquad.zip
    
    raw_dir = "/content/data/raw"
else:
    raw_dir = os.path.join(os.getcwd(), "data", "raw")
    if not os.path.exists(raw_dir):
        raw_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "raw")

json_files = sorted(glob.glob(os.path.join(raw_dir, "*", "*.json")))
print(f"JSON files found ({len(json_files)} total):")
for f in json_files:
    print(f"  - {f}")

Dataset URL: https://www.kaggle.com/datasets/incidelen/medturkquad
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
JSON files found (4 total):
  - /content/data/raw/MedTurkQuAD/MedTurkQuAD.json
  - /content/data/raw/MedTurkQuAD/test.json
  - /content/data/raw/MedTurkQuAD/train.json
  - /content/data/raw/MedTurkQuAD/validation.json


## 3. Load and inspect train.json

In [4]:
# Load train.json
train_file = [f for f in json_files if 'train.json' in f][0]
with open(train_file, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Convert to DataFrame
train_records = []
for item in train_data:
    train_records.append({
        'context': item['context'],
        'question': item['question'],
        'answer_text': item['answers']['text'][0],
        'answer_start': item['answers']['answer_start'][0]
    })

df_train = pd.DataFrame(train_records)

In [5]:
# Compute text lengths
df_train['context_length'] = df_train['context'].str.len()
df_train['question_length'] = df_train['question'].str.len()
df_train['answer_length'] = df_train['answer_text'].str.len()

print("Shape:", df_train.shape)
print("\nColumn names:", df_train.columns.tolist())
print("\nNull values:")
print(df_train.isnull().sum())
print("\nText length statistics:")
print(df_train[['context_length', 'question_length', 'answer_length']].describe())
print("\nMean lengths:")
print(f"  Context: {df_train['context_length'].mean():.1f} characters")
print(f"  Question: {df_train['question_length'].mean():.1f} characters")
print(f"  Answer: {df_train['answer_length'].mean():.1f} characters")
print("\nFirst 5 rows:")
df_train.head()

Shape: (6560, 7)

Column names: ['context', 'question', 'answer_text', 'answer_start', 'context_length', 'question_length', 'answer_length']

Null values:
context            0
question           0
answer_text        0
answer_start       0
context_length     0
question_length    0
answer_length      0
dtype: int64

Text length statistics:
       context_length  question_length  answer_length
count     6560.000000      6560.000000    6560.000000
mean       900.372713        58.878811      37.665701
std        354.598036        20.709553      40.743356
min        195.000000         9.000000       0.000000
25%        662.000000        45.000000      12.000000
50%        884.000000        58.000000      24.000000
75%       1073.500000        72.000000      47.000000
max       2837.000000       176.000000     396.000000

Mean lengths:
  Context: 900.4 characters
  Question: 58.9 characters
  Answer: 37.7 characters

First 5 rows:


,context,question,answer_text,answer_start,context_length,question_length,answer_length
0,"Apse (apse; abscess; abscessus), irinli yangı ...",Apse nedir ve nasıl oluşur?,"irinli yangı bölgesinde doku erimesi vardır, o...",33,531,27,73
1,"Apse (apse; abscess; abscessus), irinli yangı ...",Apse genellikle neyin neden olduğu bir yangı t...,piyojen bakterilerin,119,531,53,20
2,"Apse (apse; abscess; abscessus), irinli yangı ...",Apse'nin belirtileri nelerdir?,"ağrı, kızarıklık ve şişikler",251,531,30,28
3,"Apse (apse; abscess; abscessus), irinli yangı ...",Apse'nin tedavisi nedir?,"drenaj, yani boşaltılması",439,531,24,25
4,"Apse (apse; abscess; abscessus), irinli yangı ...",Cerrahi girişimden önce apse tedavisinde ne uy...,antibiyotik,493,531,59,11


## 4. Load and inspect validation.json

In [6]:
val_file = [f for f in json_files if 'validation.json' in f][0]
with open(val_file, 'r', encoding='utf-8') as f:
    val_data = json.load(f)

val_records = []
for item in val_data:
    val_records.append({
        'context': item['context'],
        'question': item['question'],
        'answer_text': item['answers']['text'][0],
        'answer_start': item['answers']['answer_start'][0]
    })

df_val = pd.DataFrame(val_records)

In [7]:
# Compute text lengths
df_val['context_length'] = df_val['context'].str.len()
df_val['question_length'] = df_val['question'].str.len()
df_val['answer_length'] = df_val['answer_text'].str.len()

print("Shape:", df_val.shape)
print("\nNull values:")
print(df_val.isnull().sum())
print("\nText length statistics:")
print(df_val[['context_length', 'question_length', 'answer_length']].describe())
print("\nMean lengths:")
print(f"  Context: {df_val['context_length'].mean():.1f} characters")
print(f"  Question: {df_val['question_length'].mean():.1f} characters")
print(f"  Answer: {df_val['answer_length'].mean():.1f} characters")
print("\nFirst 5 rows:")
df_val.head()

Shape: (820, 7)

Null values:
context            0
question           0
answer_text        0
answer_start       0
context_length     0
question_length    0
answer_length      0
dtype: int64

Text length statistics:
       context_length  question_length  answer_length
count      820.000000       820.000000     820.000000
mean      1061.701220        59.345122      40.541463
std        238.131672        19.764589      48.738113
min        375.000000        12.000000       1.000000
25%        931.000000        46.000000      12.000000
50%       1084.000000        59.000000      24.000000
75%       1197.000000        72.000000      50.000000
max       1669.000000       121.000000     516.000000

Mean lengths:
  Context: 1061.7 characters
  Question: 59.3 characters
  Answer: 40.5 characters

First 5 rows:


,context,question,answer_text,answer_start,context_length,question_length,answer_length
0,"Verem, bakteriyel ve bulaşıcı bir hastalıktır....",Verem nedir?,bakteriyel ve bulaşıcı bir hastalıktır,7,542,12,38
1,"Verem, bakteriyel ve bulaşıcı bir hastalıktır....",Vereme ne neden olur?,Mycobacterium tuberculosis mikrobu,47,542,21,34
2,"Verem, bakteriyel ve bulaşıcı bir hastalıktır....",Verem nasıl bulaşır?,Verem hastasının çevreye tükürdüğü balgamı ya ...,94,542,20,95
3,"Verem, bakteriyel ve bulaşıcı bir hastalıktır....",Veremin belirtileri nelerdir?,"ateş, titreme, gece terlemesi, iştahsızlık, ki...",210,542,29,67
4,"Verem, bakteriyel ve bulaşıcı bir hastalıktır....",Verem hastalığının tedavisinde hangi antibiyot...,"rifampisin, izoniazid, pirazinamid ve etambutol",461,542,63,47


## 5. Load and inspect test.json

In [8]:
# Load test.json
test_file = [f for f in json_files if 'test.json' in f][0]
with open(test_file, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

test_records = []
for item in test_data:
    test_records.append({
        'context': item['context'],
        'question': item['question'],
        'answer_text': item['answers']['text'][0],
        'answer_start': item['answers']['answer_start'][0]
    })

df_test = pd.DataFrame(test_records)

In [9]:
# Compute text lengths
df_test['context_length'] = df_test['context'].str.len()
df_test['question_length'] = df_test['question'].str.len()
df_test['answer_length'] = df_test['answer_text'].str.len()

print("Shape:", df_test.shape)
print("\nNull values:")
print(df_test.isnull().sum())
print("\nText length statistics:")
print(df_test[['context_length', 'question_length', 'answer_length']].describe())
print("\nMean lengths:")
print(f"  Context: {df_test['context_length'].mean():.1f} characters")
print(f"  Question: {df_test['question_length'].mean():.1f} characters")
print(f"  Answer: {df_test['answer_length'].mean():.1f} characters")
print("\nFirst 5 rows:")
df_test.head()

Shape: (820, 7)

Null values:
context            0
question           0
answer_text        0
answer_start       0
context_length     0
question_length    0
answer_length      0
dtype: int64

Text length statistics:
       context_length  question_length  answer_length
count      820.000000       820.000000     820.000000
mean      1258.042683        62.550000      35.931707
std        258.112983        22.761248      35.131606
min        390.000000         9.000000       1.000000
25%       1139.000000        47.000000      12.000000
50%       1280.000000        61.000000      25.500000
75%       1398.000000        75.000000      45.000000
max       1766.000000       150.000000     230.000000

Mean lengths:
  Context: 1258.0 characters
  Question: 62.5 characters
  Answer: 35.9 characters

First 5 rows:


,context,question,answer_text,answer_start,context_length,question_length,answer_length
0,"Tıp, bir hastaya bakma, teşhis, prognoz, önlem...",Tıp nedir?,"bir hastaya bakma, teşhis, prognoz, önleme, te...",5,662,10,151
1,"Tıp, bir hastaya bakma, teşhis, prognoz, önlem...",Tıp neyi kapsar?,çeşitli sağlık uygulamalarını,257,662,16,29
2,"Tıp, bir hastaya bakma, teşhis, prognoz, önlem...",Çağdaş tıp hangi alanları uygular?,"biyomedikal bilimleri, biyomedikal araştırmala...",376,662,34,79
3,"Tıp, bir hastaya bakma, teşhis, prognoz, önlem...",Çağdaş tıpta tedavi yolları nelerdir?,"psikoterapi, harici ateller ve traksiyon, tıbb...",525,662,37,97
4,"Tıp, bir hastaya bakma, teşhis, prognoz, önlem...",Tıp hangi yollarla sağlığı korumak ve iyileşti...,hastalıkların önlenmesi ve tedavisi,163,662,67,35


## 6. Summary

In [10]:
print("Dataset overview:")
print(f"  Train: {df_train.shape[0]} samples")
print(f"  Validation: {df_val.shape[0]} samples")
print(f"  Test: {df_test.shape[0]} samples")
print(f"  Total: {df_train.shape[0] + df_val.shape[0] + df_test.shape[0]} samples")

print("\nMean text lengths across all splits:")
print(f"  Context: ~{(df_train['context_length'].mean() + df_val['context_length'].mean() + df_test['context_length'].mean())/3:.0f} characters")
print(f"  Question: ~{(df_train['question_length'].mean() + df_val['question_length'].mean() + df_test['question_length'].mean())/3:.0f} characters")
print(f"  Answer: ~{(df_train['answer_length'].mean() + df_val['answer_length'].mean() + df_test['answer_length'].mean())/3:.0f} characters")

print("\nObservations:")
print("  - Contexts are relatively short medical passages")
print("  - Questions are concise")
print("  - Answers are short spans extracted from context")
print("  - No null values detected")
print("  - All splits have consistent structure")

Dataset overview:
  Train: 6560 samples
  Validation: 820 samples
  Test: 820 samples
  Total: 8200 samples

Mean text lengths across all splits:
  Context: ~1073 characters
  Question: ~60 characters
  Answer: ~38 characters

Observations:
  - Contexts are relatively short medical passages
  - Questions are concise
  - Answers are short spans extracted from context
  - No null values detected
  - All splits have consistent structure
